In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
base_url = "https://jobs.xomdata.com"


skills_to_track = [
    "Python", "SQL", "Power BI", "Tableau", "Excel", 
    "Pandas", "NumPy", "Spark", "AWS", "Azure", 
    "Docker", "ETL", "Machine Learning", "R", "Git"
]


start_page = 1
end_page = 51


all_extracted_jobs = []

## THU THẬP LINK CÔNG VIỆC TỪ CÁC TRANG ##

In [ ]:


job_links_pool = []

for page in range(start_page, end_page + 1):

    page_url = f"{base_url}/?page={page}"
    print(f"-> Đang quét trang {page}: {page_url}")
    
    try:
        response = requests.get(page_url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"   [!] Không thể truy cập trang {page} (Status: {response.status_code})")
            continue
            
        soup = BeautifulSoup(response.text, "html.parser")
        links = soup.find_all("a", href=True)
        
        page_job_count = 0
        for link in links:
            href = link["href"]
      
            if href.startswith("/jobs/"):
                raw_text = link.get_text(" ", strip=True)
                full_url = base_url + href
                
        
                if full_url not in [item['url'] for item in job_links_pool]:
                    job_links_pool.append({
                        "url": full_url,
                        "raw_text": raw_text
                    })
                    page_job_count += 1
                    
        print(f"   [✓] Tìm thấy {page_job_count} vị trí tại trang {page}")
        time.sleep(1) 
        
    except Exception as e:
        print(f"   [X] Lỗi khi xử lý trang {page}: {e}")

print(f"\n=> TỔNG CỘNG: Đã thu thập được {len(job_links_pool)} link công việc sẵn sàng bóc tách.")

--- BƯỚC 2: ĐANG THU THẬP LINK CÔNG VIỆC TỪ CÁC TRANG ---
-> Đang quét trang 1: https://jobs.xomdata.com/?page=1
   [✓] Tìm thấy 20 vị trí tại trang 1
-> Đang quét trang 2: https://jobs.xomdata.com/?page=2
   [✓] Tìm thấy 20 vị trí tại trang 2
-> Đang quét trang 3: https://jobs.xomdata.com/?page=3
   [✓] Tìm thấy 20 vị trí tại trang 3
-> Đang quét trang 4: https://jobs.xomdata.com/?page=4
   [✓] Tìm thấy 20 vị trí tại trang 4
-> Đang quét trang 5: https://jobs.xomdata.com/?page=5
   [✓] Tìm thấy 20 vị trí tại trang 5
-> Đang quét trang 6: https://jobs.xomdata.com/?page=6
   [✓] Tìm thấy 20 vị trí tại trang 6
-> Đang quét trang 7: https://jobs.xomdata.com/?page=7
   [✓] Tìm thấy 20 vị trí tại trang 7
-> Đang quét trang 8: https://jobs.xomdata.com/?page=8
   [✓] Tìm thấy 19 vị trí tại trang 8
-> Đang quét trang 9: https://jobs.xomdata.com/?page=9
   [✓] Tìm thấy 20 vị trí tại trang 9
-> Đang quét trang 10: https://jobs.xomdata.com/?page=10
   [✓] Tìm thấy 20 vị trí tại trang 10
-> Đang q

##  TIẾN HÀNH BÓC TÁCH CHI TIẾT TỪNG VỊ TRÍ ##

In [ ]:


for idx, job_item in enumerate(job_links_pool, 1):
    url = job_item["url"]
    raw_text = job_item["raw_text"]
    
    print(f"[{idx}/{len(job_links_pool)}] Đang xử lý: {url}")

    title_part = re.split(r'\bMới\b', raw_text)[0].strip()
    job_title = title_part.replace("Tuyển dụng ", "") 
    
    salary = "Thỏa thuận"
    if "thoả thuận" in raw_text.lower() or "thỏa thuận" in raw_text.lower():
        salary = "Thỏa thuận"
    else:

        salary_match = re.search(r'(\d+\s*-\s*\d+\s*triệu|\d+\s*triệu)', raw_text.lower())
        if salary_match:
            salary = salary_match.group(1)

  
    location = "Toàn quốc / Chưa rõ"
    if "hà nội" in raw_text.lower() or "[hn]" in raw_text.lower():
        location = "Hà Nội"
    elif "hồ chí minh" in raw_text.lower() or "tp.hcm" in raw_text.lower() or "tp.hồ chí minh" in raw_text.lower():
        location = "TP. Hồ Chí Minh"
    elif "đà nẵng" in raw_text.lower() or "[đn]" in raw_text.lower():
        location = "Đà Nẵng"


    time_match = re.search(r'(\d+\s+(giờ|phút|ngày)\s+trước)', raw_text)
    job_time = time_match.group(1) if time_match else "Vừa đăng"


    detected_skills = []
    try:
        detail_res = requests.get(url, headers=headers, timeout=10)
        if detail_res.status_code == 200:
            detail_html = detail_res.text.lower()
            
        
            for skill in skills_to_track:
                if skill.lower() in detail_html:
                    detected_skills.append(skill)
        else:
            print(f"   [!] Lỗi tải trang chi tiết (Status: {detail_res.status_code})")
            
    except Exception as e:
        print(f"   [!] Không thể kết nối tới trang chi tiết: {e}")
        
   
    all_extracted_jobs.append({
        "Vị trí công việc": job_title,
        "Nơi làm việc": location,
        "Mức lương": salary,
        "Thời gian đăng": job_time,
        "Kỹ năng yêu cầu": ", ".join(detected_skills) if detected_skills else "Không rõ (Đọc JD)",
        "Đường link chi tiết": url
    })
    
   
    time.sleep(1)

print("\n[✓] Hoàn thành bóc tách toàn bộ dữ liệu!")

--- BƯỚC 3: TIẾN HÀNH BÓC TÁCH CHI TIẾT TỪNG VỊ TRÍ ---
[1/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685188
[2/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685181
[3/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685179
[4/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685176
[5/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685175
[6/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685173
[7/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685172
[8/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685171
[9/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685168
[10/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685167
[11/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685165
[12/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685161
[13/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685160
[14/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685157
[15/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685156
[16/1011] Đang xử lý: https://jobs.xomdata.com/jobs/685154
[17/1011]

In [ ]:

df_jobs_final = pd.DataFrame(all_extracted_jobs)

print("\n--- BẢNG DỮ LIỆU TỔNG HỢP ---")
display(df_jobs_final) 



--- BẢNG DỮ LIỆU TỔNG HỢP ---


,Vị trí công việc,Nơi làm việc,Mức lương,Thời gian đăng,Kỹ năng yêu cầu,Đường link chi tiết
0,(Senior) Business Intelligence (SBS Performanc...,TP. Hồ Chí Minh,Thỏa thuận,19 phút trước,"Python, SQL, Excel, ETL, Machine Learning, R",https://jobs.xomdata.com/jobs/685188
1,Junior Business Analyst / Chuyên Viên Phân Tíc...,Hà Nội,Thỏa thuận,1 giờ trước,"SQL, Excel, Machine Learning, R",https://jobs.xomdata.com/jobs/685181
2,Data Engineer,Hà Nội,Thỏa thuận,1 giờ trước,"SQL, Spark, ETL, Machine Learning, R",https://jobs.xomdata.com/jobs/685179
3,Business Analyst (All levels),TP. Hồ Chí Minh,Thỏa thuận,3 giờ trước,"Machine Learning, R, Git",https://jobs.xomdata.com/jobs/685176
4,Senior Data Science,Hà Nội,Thỏa thuận,3 giờ trước,"Python, SQL, Power BI, Tableau, Spark, Machine...",https://jobs.xomdata.com/jobs/685175
...,...,...,...,...,...,...
1006,Kỹ sư khoa học dữ liệu - Data Scientist CÔNG T...,Hà Nội,Thỏa thuận,Vừa đăng,"Python, Pandas, Machine Learning, R, Git",https://jobs.xomdata.com/jobs/684960
1007,Wind Farm Data Analyst & Technical Support Spe...,TP. Hồ Chí Minh,Thỏa thuận,Vừa đăng,"Python, SQL, Tableau, Excel, Machine Learning, R",https://jobs.xomdata.com/jobs/7367
1008,Data Scientist Công ty TNHH T.M.G 20 - 30 triệ...,TP. Hồ Chí Minh,20 - 30 triệu,Vừa đăng,"Python, SQL, Excel, Pandas, NumPy, Machine Lea...",https://jobs.xomdata.com/jobs/684950
1009,Business Analyst CÔNG TY CỔ PHẦN CÔNG NGHỆ QUỐ...,Hà Nội,10 - 20 triệu,Vừa đăng,"Machine Learning, R",https://jobs.xomdata.com/jobs/684986


## LƯU BẢNG THÀNH FILE CSV ##

In [ ]:


file_name = "tuyen_dung.csv"


df_jobs_final.to_csv(file_name, index=False, encoding="utf-8-sig")

print(f"[✓] Đã lưu dữ liệu thành công vào file: {file_name}")

[✓] Đã lưu dữ liệu thành công vào file: tuyen_dung.csv
